In [ ]:
import pandas as pd
import os
from pathlib import Path

DATA_LOCAL = Path('/home/igor/igor_repos/scaling_laws/data_local')

df_missing = pd.read_csv('/home/igor/igor_repos/scaling_laws/Scaling-up-measurement-noise-scaling-laws/analysis/2026-04-08_veryfing_data_correctness/missing_mi_on_disk.csv')
print(f'Missing MI rows to check: {len(df_missing)}')

In [70]:
# Check which missing rows now have MI files in data_local
def mi_path(row):
    suffix = '_geneformer' if row['algorithm'] == 'Geneformer' else ''
    stem = f"Y_{row['signal']}_{row['quality']}{suffix}"
    return DATA_LOCAL / row['dataset'] / str(row['size']) / str(row['quality']) / 'results' / row['algorithm'] / 'model' / 'MI' / str(row['seed']) / stem / 'lmi_mutual_information.txt'

df_missing['path'] = df_missing.apply(mi_path, axis=1)
df_missing['done'] = df_missing['path'].apply(lambda p: os.path.exists(p))

n_done = df_missing['done'].sum()
n_still_missing = len(df_missing) - n_done
print(f'Done in data_local:    {n_done}/{len(df_missing)}')
print(f'Still missing:         {n_still_missing}/{len(df_missing)}')

Done in data_local:    18/115
Still missing:         97/115


In [71]:
# Breakdown by dataset x algorithm
join_cols = ['dataset', 'algorithm']

print('=== Done ===')
done = df_missing[df_missing['done']]
if len(done):
    display(done.groupby(join_cols).size().unstack(fill_value=0))
else:
    print('None')

print('\n=== Still missing ===')
still = df_missing[~df_missing['done']]
if len(still):
    display(still.groupby(join_cols).size().unstack(fill_value=0))
else:
    print('None')

=== Done ===


algorithm,Geneformer,PCA,RandomProjection,SCVI
dataset,,,,
shendure,1,10,4,3



=== Still missing ===


algorithm,Geneformer,PCA,RandomProjection,SCVI
dataset,,,,
shendure,24,60,1,12


In [72]:

import scanpy as sc
import numpy as np

# Load the h5ad file
h5ad_path = "/home/igor/igor_repos/scaling_laws/data_local/shendure/16681/0.0465384/preprocessed/preprocessed.h5ad"
adata = sc.read_h5ad(h5ad_path)

print("=" * 60)
print("H5AD FILE STRUCTURE INSPECTION")
print("=" * 60)
print(f"\nFile: {h5ad_path}")
print(f"\nShape (cells x genes): {adata.shape}")
print(f"\nNumber of observations (cells): {adata.n_obs}")
print(f"Number of variables (genes): {adata.n_vars}")

print("\n" + "=" * 60)
print("OBS (OBSERVATION/CELL) COLUMNS:")
print("=" * 60)
print(f"Number of obs columns: {len(adata.obs.columns)}")
print("\nColumn names and types:")
for col in adata.obs.columns:
    print(f"  - {col}: {adata.obs[col].dtype}")

print("\n" + "=" * 60)
print("VAR (VARIABLE/GENE) COLUMNS:")
print("=" * 60)
print(f"Number of var columns: {len(adata.var.columns)}")
print("\nColumn names and types:")
for col in adata.var.columns:
    print(f"  - {col}: {adata.var[col].dtype}")

print("\n" + "=" * 60)
print("LAYERS:")
print("=" * 60)
if adata.layers:
    print(f"Number of layers: {len(adata.layers)}")
    for layer_name in adata.layers.keys():
        layer = adata.layers[layer_name]
        print(f"  - {layer_name}: shape {layer.shape}, dtype {layer.dtype}")
else:
    print("No layers present")

print("\n" + "=" * 60)
print("X (MAIN DATA MATRIX):")
print("=" * 60)
print(f"Type: {type(adata.X)}")
print(f"Shape: {adata.X.shape}")
print(f"Dtype: {adata.X.dtype}")
if hasattr(adata.X, 'format'):
    print(f"Format: {adata.X.format}")

print("\n" + "=" * 60)
print("OBSM (OBSERVATION EMBEDDINGS):")
print("=" * 60)
if adata.obsm:
    print(f"Number of embeddings: {len(adata.obsm.keys())}")
    for key in adata.obsm.keys():
        print(f"  - {key}: shape {adata.obsm[key].shape}")
else:
    print("No embeddings")

print("\n" + "=" * 60)
print("VARM (VARIABLE EMBEDDINGS):")
print("=" * 60)
if adata.varm:
    print(f"Number of embeddings: {len(adata.varm.keys())}")
    for key in adata.varm.keys():
        print(f"  - {key}: shape {adata.varm[key].shape}")
else:
    print("No embeddings")

print("\n" + "=" * 60)
print("OBSP (OBSERVATION PAIRWISE):")
print("=" * 60)
if adata.obsp:
    print(f"Number of pairwise matrices: {len(adata.obsp.keys())}")
    for key in adata.obsp.keys():
        print(f"  - {key}: shape {adata.obsp[key].shape}")
else:
    print("No pairwise matrices")

print("\n" + "=" * 60)
print("UNS (UNSTRUCTURED METADATA):")
print("=" * 60)
if adata.uns:
    print(f"Number of uns keys: {len(adata.uns.keys())}")
    print("Keys:", list(adata.uns.keys())[:20])  # Show first 20
else:
    print("No unstructured metadata")


H5AD FILE STRUCTURE INSPECTION

File: /home/igor/igor_repos/scaling_laws/data_local/shendure/16681/0.0465384/preprocessed/preprocessed.h5ad

Shape (cells x genes): (16681, 45525)

Number of observations (cells): 16681
Number of variables (genes): 45525

OBS (OBSERVATION/CELL) COLUMNS:
Number of obs columns: 25

Column names and types:
  - assay: category
  - assay_ontology_term_id: category
  - author_cell_type: category
  - author_day: category
  - author_experimental_id: category
  - author_major_cell_cluster: category
  - author_somite_count: category
  - cell_type: category
  - cell_type_ontology_term_id: category
  - development_stage: category
  - development_stage_ontology_term_id: category
  - disease: category
  - disease_ontology_term_id: category
  - donor_id: category
  - is_primary_data: bool
  - n_counts: float32
  - observation_joinid: object
  - self_reported_ethnicity: category
  - self_reported_ethnicity_ontology_term_id: category
  - sex: category
  - sex_ontology_te

In [1]:

import h5py
import json

path = '/home/igor/igor_repos/scaling_laws/data_local/shendure/16681/0.0465384/preprocessed/preprocessed.h5ad'

with h5py.File(path, 'r') as f:
    # Print all top-level keys
    print("Top-level keys:", list(f.keys()))
    
    # Print obs column names
    if 'obs' in f:
        print("\nobs keys:", list(f['obs'].keys()))
        # Try to read a few values from each column
        for key in list(f['obs'].keys())[:20]:
            try:
                data = f['obs'][key]
                if hasattr(data, 'shape'):
                    print(f"  {key}: shape={data.shape}, dtype={data.dtype}, first few={data[:3]}")
                else:
                    print(f"  {key}: {type(data)}")
            except:
                print(f"  {key}: (could not read)")
    
    # Print var column names
    if 'var' in f:
        print("\nvar keys:", list(f['var'].keys()))
        for key in list(f['var'].keys())[:10]:
            try:
                data = f['var'][key]
                if hasattr(data, 'shape'):
                    print(f"  {key}: shape={data.shape}, dtype={data.dtype}, first few={data[:3]}")
            except:
                print(f"  {key}: (could not read)")
    
    # Print obsm keys
    if 'obsm' in f:
        print("\nobsm keys:", list(f['obsm'].keys()))
    
    # Print layers
    if 'layers' in f:
        print("\nlayers keys:", list(f['layers'].keys()))
    
    # Print X shape
    if 'X' in f:
        x = f['X']
        print(f"\nX: keys={list(x.keys()) if hasattr(x, 'keys') else 'array'}")
        if hasattr(x, 'shape'):
            print(f"  shape: {x.shape}")

    # Print uns keys
    if 'uns' in f:
        print("\nuns keys:", list(f['uns'].keys()))


Top-level keys: ['X', 'layers', 'obs', 'obsm', 'obsp', 'uns', 'var', 'varm', 'varp']

obs keys: ['_index', 'assay', 'assay_ontology_term_id', 'author_cell_type', 'author_day', 'author_experimental_id', 'author_major_cell_cluster', 'author_somite_count', 'cell_type', 'cell_type_ontology_term_id', 'development_stage', 'development_stage_ontology_term_id', 'disease', 'disease_ontology_term_id', 'donor_id', 'is_primary_data', 'n_counts', 'observation_joinid', 'self_reported_ethnicity', 'self_reported_ethnicity_ontology_term_id', 'sex', 'sex_ontology_term_id', 'suspension_type', 'tissue', 'tissue_ontology_term_id', 'tissue_type']
  _index: shape=(16681,), dtype=object, first few=[b'run_4_P2-01A.AAGGCTACTTTCTTCCGGT-0'
 b'run_4_P2-01B.ACTGGACCTTTATTCTGAG-0'
 b'run_4_P2-01D.ATGCATTCATGACGAAGCGT-0']
  assay: <class 'h5py._hl.group.Group'>
  assay_ontology_term_id: <class 'h5py._hl.group.Group'>
  author_cell_type: <class 'h5py._hl.group.Group'>
  author_day: <class 'h5py._hl.group.Group'>
  aut

In [ ]:

import h5py
import numpy as np

path = '/home/igor/igor_repos/scaling_laws/data_local/shendure/16681/0.0465384/preprocessed/preprocessed.h5ad'

with h5py.File(path, 'r') as f:
    obs = f['obs']
    
    print("=== ALL OBS COLUMNS WITH DETAILED VALUES ===\n")
    
    # Get all obs keys
    obs_keys = list(obs.keys())
    
    for key in obs_keys:
        data = obs[key]
        print(f"\n{key}:")
        
        if isinstance(data, h5py.Group):
            # It's a categorical column stored as a group
            print(f"  Type: Categorical (Group)")
            
            # Check what keys are in the group
            group_keys = list(data.keys())
            print(f"  Group keys: {group_keys}")
            
            if 'codes' in data and 'categories' in data:
                codes = data['codes'][:]
                categories = data['categories'][:]
                print(f"  codes shape: {codes.shape}, dtype: {codes.dtype}")
                print(f"  categories shape: {categories.shape}, dtype: {categories.dtype}")
                print(f"  categories: {categories}")
                print(f"  first 10 values (by code): {codes[:10]}")
                # Show actual values
                actual_values = categories[codes[:10]]
                print(f"  first 10 actual values: {actual_values}")
                print(f"  unique values count: {len(np.unique(codes))}")
        else:
            # It's an array
            print(f"  Type: Array")
            print(f"  dtype: {data.dtype}")
            print(f"  shape: {data.shape}")
            print(f"  first 10 values: {data[:10]}")
            if data.dtype == object or data.dtype.kind == 'U':
                print(f"  unique values: {np.unique(data[:])}")
